In [ ]:
import pandas as pd

from tzlocal import get_localzone
from datetime import datetime
import plotly.graph_objects as go
import warnings
warnings.simplefilter('ignore')
warnings.filterwarnings('ignore')
from tqdm import tqdm
tqdm.pandas()

import train_model as tm
from train_model import ModelFunc
import data_processing as dp
from data_loader import load_data_at_start_date, load_data_by_period
from features import FeatureEngineering


In [ ]:
period= -(datetime.now() - datetime(2018, 1, 1)).days
load_data_at_start_date(['BTC-USD'], period, '1d', 'crypto_data')
data = dp.get_data('crypto_data', 'BTC-USD', compress=False)
data = tm.standard_scaler(data)
# display(data)

In [ ]:
fe_params = {
    'emaf': 20,
    'emam': 100,
    'emas': 150,
    'rsi': 14,
    'macd': [12, 26, 9],
 }

fe = FeatureEngineering(fe_params)

In [ ]:
lag_periods = 3
features_to_trend = ['Open', 'High', 'Low', 'Close', 'Volume']
data = fe.clear_invalid_targets(fe.add_target(fe.enrich_with_indicators(data), lag_periods))
data_with_trend, new_trend_features = fe.create_trend_features(data, features_to_trend, lag_periods) 
data = data_with_trend[new_trend_features + ['Target']]


In [ ]:
split_param = {
    'last_test_month_cnt': 1,
    'last_val_month_cnt': 2,
}

train_data, val_data, test_data = tm.split_data_by_date2(data, split_param)

X_train, y_train = tm.split_by_features_and_target_variables(train_data, new_trend_features)
X_val, y_val = tm.split_by_features_and_target_variables(val_data, new_trend_features)
X_test, y_test = tm.split_by_features_and_target_variables(test_data, new_trend_features)

print(f"Train size: {len(X_train)}, Val size: {len(X_val)}, Test size: {len(X_test)}")